In [1]:
# %%
# ============================================================
# ☀️ Build 3-Day EVE Irradiance Targets (6.5 nm)
# ============================================================
# • Reads "sdo_eve_ssi_1nm_l3.csv"
# • Extracts irradiance at 6.5 nm for each flare start
# • Finds nearest EVE values on T₀, T₀ + 1 day, T₀ + 2 days
# • Outputs: flare_euv_targets_3day.csv
# ============================================================

import pandas as pd
from datetime import datetime, timedelta
import numpy as np

In [3]:
# ============================================================
# 🔧  Config
# ============================================================

EVE_FILE = "sdo_eve_ssi_1nm_l3 (2).csv"

# Your 10 selected flare start times
FLARES = [
    ("AR11158_X2.2","2011-02-15 01:44:00"),
    ("AR11429_X5.4","2012-03-07 00:02:00"),
    ("AR11520_X1.4","2012-07-12 15:37:00"),
    ("AR12158_X1.6","2014-09-10 17:21:00"),
    ("AR12192_X3.1","2014-10-24 21:07:00"),
    ("AR11158_M6.6","2011-02-13 17:28:00"),
    ("AR11261_M9.3","2011-07-30 02:04:00"),
    ("AR11429_M6.3","2012-03-09 03:22:00"),
    ("AR11719_M6.5","2013-04-11 06:55:00"),
    ("AR12036_M7.3","2014-04-18 12:31:00"),
]

print(f"✅ Loaded {len(FLARES)} flare start times")

✅ Loaded 10 flare start times


In [4]:
# ============================================================
# ☀️ 1. Load & clean EVE L3 CSV
# ============================================================

df = pd.read_csv(EVE_FILE)

# Standardize column names
df.columns = [c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "") for c in df.columns]
time_col = [c for c in df.columns if "time" in c][0]
df["timestamp_utc"] = pd.to_datetime(df[time_col], errors="coerce")

# Keep only wavelength ≈ 6.5 nm
df_65 = df[np.isclose(df["wavelength_nm"], 6.5, atol=0.05)].copy()
df_65 = df_65.rename(columns={"irradiance_w/m^2/nm": "irradiance_6p5nm"})
df_65 = df_65.dropna(subset=["irradiance_6p5nm","timestamp_utc"]).sort_values("timestamp_utc").reset_index(drop=True)

print("✅ EVE 6.5 nm records:", len(df_65))
print(df_65.head(3))


✅ EVE 6.5 nm records: 5637
  time_yyyy-mm-dd't'hh:mm:ss  wavelength_nm  irradiance_6p5nm  stdev_w/m^2/nm  \
0        2010-04-30T12:00:34            6.5          0.000050        0.000192   
1        2010-05-01T12:00:34            6.5          0.000050        0.000186   
2        2010-05-02T12:00:34            6.5          0.000051        0.000226   

   precision  accuracy       timestamp_utc  
0   0.000705  0.072377 2010-04-30 12:00:34  
1   0.000703  0.069729 2010-05-01 12:00:34  
2   0.000709  0.069954 2010-05-02 12:00:34  


In [5]:
# ============================================================
# ☀️ 2.  Build 3-day targets for each flare
# ============================================================

records = []

for flare_name, flare_start in FLARES:
    t0 = pd.to_datetime(flare_start)
    vals = []
    for i in range(3):  # T0, T0 + 1 d, T0 + 2 d
        target_time = t0 + timedelta(days=i)
        idx = (df_65["timestamp_utc"] - target_time).abs().idxmin()
        match = df_65.loc[idx]
        vals.append(match["irradiance_6p5nm"])
    records.append((flare_name, t0, *vals))

target_df = pd.DataFrame(records, columns=["flare_id","flare_time","EUV_T0","EUV_T1","EUV_T2"])
target_df = target_df.sort_values("flare_time").reset_index(drop=True)

In [6]:
# ============================================================
# ☀️ 3. Save target CSV
# ============================================================

out_csv = "flare_euv_targets_3day.csv"
target_df.to_csv(out_csv, index=False)

print(f"💾 Saved → {out_csv}")
print(target_df)

💾 Saved → flare_euv_targets_3day.csv
       flare_id          flare_time    EUV_T0    EUV_T1    EUV_T2
0  AR11158_M6.6 2011-02-13 17:28:00  0.000060  0.000062  0.000063
1  AR11158_X2.2 2011-02-15 01:44:00  0.000063  0.000062  0.000064
2  AR11261_M9.3 2011-07-30 02:04:00  0.000062  0.000064  0.000067
3  AR11429_X5.4 2012-03-07 00:02:00  0.000084  0.000080  0.000083
4  AR11429_M6.3 2012-03-09 03:22:00  0.000083  0.000085  0.000081
5  AR11520_X1.4 2012-07-12 15:37:00  0.000086  0.000080  0.000080
6  AR11719_M6.5 2013-04-11 06:55:00  0.000097  0.000095  0.000090
7  AR12036_M7.3 2014-04-18 12:31:00  0.000113  0.000113  0.000114
8  AR12158_X1.6 2014-09-10 17:21:00  0.000000  0.000000  0.000000
9  AR12192_X3.1 2014-10-24 21:07:00  0.000000  0.000000  0.000000


In [7]:
df_65[df_65["timestamp_utc"] > "2014-08-01"].head(10)


,time_yyyy-mm-dd't'hh:mm:ss,wavelength_nm,irradiance_6p5nm,stdev_w/m^2/nm,precision,accuracy,timestamp_utc
1547,2014-08-01T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-01 12:00:35
1548,2014-08-02T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-02 12:00:35
1549,2014-08-03T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-03 12:00:35
1550,2014-08-04T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-04 12:00:35
1551,2014-08-05T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-05 12:00:35
1552,2014-08-06T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-06 12:00:35
1553,2014-08-07T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-07 12:00:35
1554,2014-08-08T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-08 12:00:35
1555,2014-08-09T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-09 12:00:35
1556,2014-08-10T12:00:35,6.5,0.0,NaN,NaN,NaN,2014-08-10 12:00:35
